In [0]:
df_json = spark.read.format("json").load("/Volumes/workspace/default/raw_data_volume/big_box_home_improvement_dataset.json")

In [0]:
df_csv = spark.read.format("csv") \
.option("header", "true") \
.option("inferSchema", "true") \
.load("/Volumes/workspace/default/raw_data_volume/big_box_home_improvement_dataset.csv")

In [0]:
# Re-read JSON with multiLine option to fix parsing
df_json = spark.read.format("json").option("multiLine", "true").load("/Volumes/workspace/default/raw_data_volume/big_box_home_improvement_dataset.json")
display(df_json)

brand,bulk_discount_applied_%,department,pro_program,project_type,purchase_date,quantity,retailer,sku,store_location,subtotal_before_discount,total_after_discount,transaction_id,unit_price
Behr,5,Electrical,Contractors' Warehouse CBC,Electrical Upgrade,2024-09-10 02:15:59,20,Do it Best,ELE-382095,"South Joyceburgh, AS",3746.2,3558.89,DO-20240910-11568,187.31
Philips,15,Drywall & Insulation,Lowe's MVPs Pro Rewards,Electrical Upgrade,2024-11-06 07:02:37,75,Lowe's,DRY-123851,"New Scott, MS",2241.0,1904.85,LOW-20241106-63550,29.88
null,25,null,Contractors' Warehouse CBC,Drywall Patch,2024-05-16 12:50:33,60,Harbor Freight Tools,ELE-160787,"North David, TX",29830.8,22373.1,HAR-20240516-41194,497.18
Makita,15,Outdoor & Garden,Ace Rewards,Landscaping,2024-11-10 05:37:10,10,Lowe's,OUT-878292,"Floresfort, AK",1551.8,1319.03,LOW-20241110-17341,155.18
LG,0,Tools & Hardware,None,Irrigation Setup,2024-07-24 16:02:53,1,Harbor Freight Tools,TOO-526831,"North Margaretside, IL",799.08,799.08,HAR-20240724-68170,799.08
Kobalt,18,Lumber & Building Materials,Ace Rewards,Electrical Upgrade,2024-03-17 09:34:47,30,Contractors' Warehouse,LUM-453299,"Burketown, PR",740.4,607.13,CON-20240317-39471,24.68
Milwaukee,5,null,Do it Best Rewards,Roof Repair,2025-07-23 18:36:47,12,Ace Hardware,KIT-784519,"Port Dominiqueview, FL",23876.28,22682.47,ACE-20250723-12817,1989.69
Toro,0,Flooring,Lowe's MVPs Pro Rewards,Electrical Upgrade,2024-07-12 19:33:13,295,Ace Hardware,FLO-136466,"Beckermouth, MA",1109.2,1109.2,ACE-20240712-93941,3.76
Whirlpool,10,Doors & Windows,None,Flooring Install,2024-05-16 17:05:00,8,Lowe's,DOO-950203,null,3920.08,3528.07,LOW-20240516-70955,490.01
DEWALT,0,Outdoor & Garden,Home Depot Pro Xtra,HVAC Replacement,2024-05-06 08:55:48,1,Menards,OUT-491404,"Brittanyborough, NE",1340.35,1340.35,MEN-20240506-54477,1340.35


In [0]:
from pyspark.sql.functions import col, upper, sum

In [0]:
#1. BRONZE LAYER (Read Raw Data)

#Read JSON dataset
df_json_raw = spark.read.format("json") .option("multiLine", "true").load("/Volumes/workspace/default/raw_data_volume/big_box_home_improvement_dataset.json")
#Read CSV dataset
df_csv_raw = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/Volumes/workspace/default/raw_data_volume/big_box_home_improvement_dataset.csv")

In [0]:
#2. SILVER LAYER (Clean & Transform)
df_json_silver = df_json_raw.filter(col("transaction_id").isNotNull())
df_csv_silver = df_csv_raw.withColumn("department_upper", upper(col("department"))).filter(col("transaction_id").isNotNull())

df_json_silver.write.format("delta").mode("overwrite").saveAsTable("silver_json_data")
df_csv_silver.write.format("delta").mode("overwrite").saveAsTable("silver_csv_data")

In [0]:
#3. GOLD LAYER (Aggregate & Summarize)
#Summarize JSON: Group by date
df_json_gold = df_json_silver.groupBy("purchase_date").agg(sum("total_after_discount").alias("total_daily_revenue"))

#Summarize CSV: Group by department
df_csv_gold = df_csv_silver.groupBy("department").agg(sum("total_after_discount").alias("total_csv_revenue"))

#Save as Gold Delta Tables
df_json_gold.write.format("delta").mode("overwrite").saveAsTable("gold_json_summary") 
df_csv_gold.write.format("delta").mode("overwrite").saveAsTable("gold_csv_summary")

In [0]:
display(df_csv_gold)

department,total_csv_revenue
Lumber & Building Materials,1126101.8400000005
Appliances,2827948.2099999995
Outdoor & Garden,1736051.1799999997
Lighting & Fans,554010.7000000002
Paint,302589.07999999996
Kitchen & Bath,2943069.040000001
Electrical,822553.1799999997
HVAC,4292150.969999998
Plumbing,956141.5599999997
Drywall & Insulation,168688.84999999992


In [0]:
display(df_json_gold)

purchase_date,total_daily_revenue
2025-02-01 18:54:38,3247.02
2024-09-15 06:45:48,9579.64
2024-04-24 02:11:14,115.81
2024-05-09 02:46:07,3802.0
2024-09-11 13:45:11,3385.08
2025-07-05 13:47:31,19517.7
2024-02-23 20:11:06,23866.64
2025-07-08 05:23:17,28196.4
2025-01-09 01:04:03,115.41
2024-05-01 23:36:07,3051.18


In [0]:
%sql 
SELECT department, ROUND(total_csv_revenue, 2) AS total_revenue FROM gold_csv_summary ORDER BY total_revenue DESC;

department,total_revenue
HVAC,4292150.97
Kitchen & Bath,2943069.04
Appliances,2827948.21
Outdoor & Garden,1736051.18
Doors & Windows,1663448.85
Tools & Hardware,1182225.53
Lumber & Building Materials,1126101.84
Plumbing,956141.56
Electrical,822553.18
Lighting & Fans,554010.7


In [0]:
%sql 
SELECT* FROM gold_json_summary ORDER BY total_daily_revenue DESC;
     


purchase_date,total_daily_revenue
2025-06-05 15:25:57,302151.2
2024-05-22 05:09:14,197864.55
2025-02-09 17:20:34,188542.66
2024-10-09 03:32:05,164358.68
2025-06-01 21:37:52,158441.6
2025-01-23 13:47:22,147126.11
2024-05-11 13:12:52,144392.48
2024-06-01 02:35:31,143189.25
2024-07-12 00:18:44,139318.0
2025-07-02 23:04:43,134209.8


In [0]:
%sql  SELECT* FROM gold_csv_summary

department,total_csv_revenue
Lumber & Building Materials,1126101.8400000005
Appliances,2827948.2099999995
Outdoor & Garden,1736051.1799999997
Lighting & Fans,554010.7000000002
Paint,302589.07999999996
Kitchen & Bath,2943069.040000001
Electrical,822553.1799999997
HVAC,4292150.969999998
Plumbing,956141.5599999997
Drywall & Insulation,168688.84999999992


Databricks visualization. Run in Databricks to view.